# บทที่ 4: ฟังก์ชันสูญเสียและการแพร่กระจายย้อนกลับ (Loss Function & Backpropagation)

ใน Notebook นี้ เราจะจำลองการทำงานของฟังก์ชันสูญเสียต่างๆ การคำนวณการแพร่กระจายย้อนกลับทีละขั้นตอน รวมถึงอธิบายการทำงานของตัวปรับค่า เพื่อให้เห็นภาพรวมเชิงปฏิบัติการตามทฤษฎีในหนังสือ


**ศัพท์ที่สำคัญในบทนี้:**- เวกเตอร์ (vector) — อาร์เรย์หนึ่งมิติ- เมทริกซ์ (matrix) — อาร์เรย์สองมิติ- ค่าน้ำหนัก (weight) — พารามิเตอร์ที่ปรับได้- ค่าไบแอส (bias) — ค่าเลื่อน- อินพุต (input) — ข้อมูลนำเข้า- เอาต์พุต (output) — ผลลัพธ์- ค่าสูญเสีย (loss) — วัดความคลาดเคลื่อน- เกรเดียนต์ (gradient) — ทิศทางการปรับ- ฟังก์ชันกระตุ้น (activation function) — ฟังก์ชันไม่เป็นเชิงเส้น- โครงข่ายประสาทเทียม (neural network) — โมเดลแมชชีนเลิร์นนิง

## 1. นำเข้าไลบรารีที่จำเป็น (Import Libraries)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
try:
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'],
                   capture_output=True)
except FileNotFoundError:
    pass  # เครื่องที่ไม่มี apt-get (macOS/Windows) ใช้ฟอนต์ไทยที่ติดตั้งไว้ในเครื่องแทน

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

## 2. ฟังก์ชันสูญเสีย (Loss Functions)
หัวข้อ 4.1 และ 4.2: การแปลงทฤษฎีเป็นโค้ด Python สำหรับค่าคลาดเคลื่อนกำลังสองเฉลี่ย (MSE) และครอสเอนโทรปี (Cross-Entropy)


In [ ]:
def mse_loss(y_true, y_pred):
    """
    ค่าคลาดเคลื่อนกำลังสองเฉลี่ย (MSE) สำหรับปัญหาการถดถอย
    """
    return np.mean((y_true - y_pred) ** 2)

def binary_cross_entropy(y_true, y_pred, epsilon=1e-15):
    """
    ครอสเอนโทรปีสำหรับปัญหาการจำแนกประเภท 2 คลาส คำนวณจากความน่าจะเป็น
    (มีการบวกค่า epsilon เล็กๆ เพื่อป้องกันการหา log(0) แต่วิธีนี้ยังไม่เสถียรที่สุด ดูฟังก์ชันถัดไป)
    """
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def binary_cross_entropy_from_logits(z, y_true):
    """
    ครอสเอนโทรปีคำนวณจากลอจิตโดยตรงด้วยสูตรที่มีเสถียรภาพเชิงตัวเลข (หัวข้อ 4.2.5):
    L = log(1 + exp(z)) - y*z ซึ่งไม่ล้นแม้ลอจิตมีขนาดใหญ่มาก
    """
    return np.mean(np.logaddexp(0, z) - y_true * z)

def categorical_cross_entropy(y_true, y_pred, epsilon=1e-15):
    """
    ครอสเอนโทรปีสำหรับปัญหาการจำแนกหลายคลาส
    """
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.sum(y_true * np.log(y_pred)) / y_pred.shape[0]

# --- ตัวอย่างการทดสอบฟังก์ชัน ---
y_true_reg = np.array([100.0, 150.0, 200.0])
y_pred_reg = np.array([110.0, 145.0, 190.0])
print(f"MSE: {mse_loss(y_true_reg, y_pred_reg):.4f}")

y_true_bin = np.array([1, 0, 1])
y_pred_bin = np.array([0.9, 0.1, 0.8])
print(f"ครอสเอนโทรปี (จากความน่าจะเป็น): {binary_cross_entropy(y_true_bin, y_pred_bin):.4f}")

# เทียบกับสูตรที่คำนวณจากลอจิตโดยตรง (ต้องได้ค่าเดียวกันเมื่อไม่ติดปัญหาขอบเขต)
z_bin = np.log(y_pred_bin / (1 - y_pred_bin))  # ลอจิตที่สอดคล้องกับ y_pred_bin
print(f"ครอสเอนโทรปี (จากลอจิตโดยตรง): {binary_cross_entropy_from_logits(z_bin, y_true_bin):.4f}")

# ทดสอบลอจิตขนาดใหญ่: สูตร clip ให้ log(0) จน NaN แต่สูตรจากลอจิตโดยตรงยังเสถียร
z_extreme = np.array([1000.0])
y_extreme = np.array([0.0])
print(f"ลอจิตขนาดใหญ่ (z=1000) ด้วยสูตรจากลอจิตโดยตรง: {binary_cross_entropy_from_logits(z_extreme, y_extreme):.4f} (ไม่เกิด NaN)")

## 3. กฎลูกโซ่และการแพร่กระจายย้อนกลับ (Chain Rule & Backpropagation)
หัวข้อ 4.3: จำลองการส่งผ่านสัญญาณไปข้างหน้าและการแพร่กระจายย้อนกลับด้วยหลักการหาอนุพันธ์


In [ ]:
# ฟังก์ชันกระตุ้นและอนุพันธ์ของฟังก์ชันกระตุ้น (Sigmoid)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    # อนุพันธ์ของ Sigmoid คือ sigmoid(x) * (1 - sigmoid(x))
    sx = sigmoid(x)
    return sx * (1 - sx)

# กำหนดตัวแปรและน้ำหนักเริ่มต้นตัวอย่าง (จากหัวข้อ 4.7 ตัวอย่างการคำนวณแบบละเอียด)
x = np.array([[1.0], [2.0]]) # อินพุต
y = np.array([[1.0]])        # คำตอบจริง

W1 = np.array([[0.5, 0.3], [0.2, 0.4]])
b1 = np.array([[0.1], [0.2]])

W2 = np.array([[0.6, 0.8]])
b2 = np.array([[0.3]])

learning_rate = 0.1

print("--- การส่งผ่านสัญญาณไปข้างหน้า ---")
# ชั้นที่ 1 (ชั้นซ่อน)
z1 = np.dot(W1, x) + b1
a1 = sigmoid(z1)
print(f"เอาต์พุตของชั้นซ่อน (a1):\n{a1}")

# ชั้นที่ 2 (ชั้นเอาต์พุต)
z2 = np.dot(W2, a1) + b2
y_hat = sigmoid(z2)
print(f"ผลทำนาย (y_hat): {y_hat[0][0]:.4f}")

# ค่าสูญเสีย (MSE)
loss = 0.5 * (y - y_hat)**2
print(f"ค่าสูญเสีย (MSE): {loss[0][0]:.4f}")

print("\n--- การแพร่กระจายย้อนกลับ ---")
# 1. เกรเดียนต์ของชั้นเอาต์พุต (เทียบกับ z2)
dz2 = -(y - y_hat) * sigmoid_derivative(z2)

# เกรเดียนต์ของพารามิเตอร์ฝั่งเอาต์พุต (W2, b2)
dW2 = np.dot(dz2, a1.T)
db2 = dz2

# 2. เกรเดียนต์ของชั้นซ่อน (เทียบกับ z1)
da1 = np.dot(W2.T, dz2)
dz1 = da1 * sigmoid_derivative(z1)

# เกรเดียนต์ของพารามิเตอร์ฝั่งชั้นซ่อน (W1, b1)
dW1 = np.dot(dz1, x.T)
db1 = dz1

# 3. อัปเดตพารามิเตอร์ (การไล่ลงตามเกรเดียนต์แบบปกติตามหัวข้อ 4.4)
W1_new = W1 - learning_rate * dW1
b1_new = b1 - learning_rate * db1
W2_new = W2 - learning_rate * dW2
b2_new = b2 - learning_rate * db2

print("W1 ถูกอัปเดตเป็น:\n", W1_new)
print("W2 ถูกอัปเดตเป็น:\n", W2_new)

## 4. ตัวปรับค่าพื้นฐาน (Basic Optimizers)
หัวข้อ 4.4: เปรียบเทียบ SGD, Momentum และอธิบายความแตกต่าง


In [ ]:
# สร้างพื้นผิวค่าสูญเสียจำลอง (พาราโบลา 1 มิติ) สำหรับสาธิตตัวปรับค่า
def loss_func(w):
    return w**2

def gradient(w):
    return 2*w

# กำหนดสภาวะเริ่มต้น
w_init = 10.0
learning_rate_demo = 0.1
epochs = 20

v_momentum = 0.0
beta = 0.9

w_sgd = w_init
w_momentum = w_init

history_sgd = [w_sgd]
history_momentum = [w_momentum]

for _ in range(epochs):
    # SGD ปกติ
    grad_sgd = gradient(w_sgd)
    w_sgd = w_sgd - learning_rate_demo * grad_sgd
    history_sgd.append(w_sgd)

    # โมเมนตัม
    grad_mom = gradient(w_momentum)
    v_momentum = beta * v_momentum + (1 - beta) * grad_mom # รูปแบบค่าเฉลี่ยเคลื่อนที่แบบเลขชี้กำลัง
    w_momentum = w_momentum - learning_rate_demo * v_momentum
    history_momentum.append(w_momentum)

plt.figure(figsize=(10, 5))
plt.plot(history_sgd, label='SGD', marker='o')
plt.plot(history_momentum, label='SGD with Momentum', marker='x')
plt.title("เปรียบเทียบการลู่เข้า (Convergence) ระหว่าง SGD และ Momentum")
plt.xlabel("รอบการฝึก")
plt.ylabel("ค่าน้ำหนัก (เป้าหมาย = 0)")
plt.axhline(0, color='red', linestyle='--', label='จุดต่ำสุดสัมบูรณ์')
plt.legend()
plt.show()

## 5. Adam (Adaptive Moment Estimation)
หัวข้อ 4.4.2: กลไกของ Adam ที่รวมโมเมนตัมและอัตราการเรียนรู้แบบปรับตัวไว้ด้วยกัน


In [ ]:
# สาธิตการทำงานของ Adam Optimizer ตัวหลัก
w_adam = w_init
m_t = 0.0
v_t = 0.0
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
lr_adam = 0.5 # ใช้ LR สูงๆ เพื่อให้เห็นผลลัพธ์ชัดบนโจทย์ตัวอย่าง

history_adam = [w_adam]

for t in range(1, epochs + 1):
    grad = gradient(w_adam)

    # คำนวณค่าเฉลี่ยเคลื่อนที่
    m_t = beta1 * m_t + (1 - beta1) * grad
    v_t = beta2 * v_t + (1 - beta2) * (grad**2)

    # การแก้ค่าเบี่ยงเบน (หัวใจสำคัญตอนเริ่มต้นเรียนรู้)
    m_t_hat = m_t / (1 - beta1**t)
    v_t_hat = v_t / (1 - beta2**t)

    # การอัปเดตน้ำหนัก
    w_adam = w_adam - lr_adam * m_t_hat / (np.sqrt(v_t_hat) + epsilon)
    history_adam.append(w_adam)

plt.figure(figsize=(10, 5))
plt.plot(history_adam, color='purple', label='Adam (LR=0.5)', marker='s')
plt.title("พฤติกรรมการลู่เข้าของ Adam Optimizer")
plt.xlabel("รอบการฝึก")
plt.ylabel("ค่าน้ำหนัก")
plt.axhline(0, color='red', linestyle='--')
plt.legend()
plt.show()

## 6. AdaGrad, RMSprop และ AdamW
หัวข้อ 4.4: ตัวปรับค่าที่ปรับขนาดก้าวแยกตามพารามิเตอร์ด้วยสถิติของเกรเดียนต์ยกกำลังสองคนละรูปแบบ ได้แก่ AdaGrad (ผลรวมสะสมที่ไม่ลืมอดีต), RMSprop (ค่าเฉลี่ยเคลื่อนที่แทนผลรวมสะสม) และ AdamW (แยกการลดทอนค่าน้ำหนักออกจากก้าวของ Adam) ทดสอบบนโจทย์เดียวกับ SGD, Momentum และ Adam ด้านบนเพื่อเปรียบเทียบเส้นทางการหาค่าเหมาะที่สุดโดยตรง

In [ ]:
# สาธิตการทำงานของ AdaGrad, RMSprop และ AdamW บนโจทย์ทดสอบเดียวกับ SGD/Momentum/Adam
# (ใช้ w_init, epochs, loss_func และ gradient ที่กำหนดไว้แล้วในเซลล์ก่อนหน้า)

# --- AdaGrad: ผลรวมสะสมของเกรเดียนต์ยกกำลังสอง G_t ไม่ลืมอดีต ก้าวจึงเล็กลงเรื่อย ๆ ---
w_adagrad = w_init
G_t = 0.0
epsilon_adagrad = 1e-8
lr_adagrad = 1.5  # ใช้ LR สูงกว่าฐานเพราะ G_t ที่สะสมหารก้าวให้เล็กลงเร็ว

history_adagrad = [w_adagrad]
for _ in range(epochs):
    grad = gradient(w_adagrad)
    G_t = G_t + grad**2                                          # ผลรวมสะสม (ไม่ลดลง)
    w_adagrad = w_adagrad - lr_adagrad * grad / (np.sqrt(G_t) + epsilon_adagrad)
    history_adagrad.append(w_adagrad)

# --- RMSprop: แทนผลรวมสะสมด้วยค่าเฉลี่ยเคลื่อนที่ ลดอิทธิพลของอดีตที่ไกลออกไป ---
w_rmsprop = w_init
v_t_rms = 0.0
beta_rms = 0.9       # อัตราลดทอน (ค่าตั้งต้นที่พบบ่อย)
epsilon_rms = 1e-8
lr_rmsprop = 0.3

history_rmsprop = [w_rmsprop]
for _ in range(epochs):
    grad = gradient(w_rmsprop)
    v_t_rms = beta_rms * v_t_rms + (1 - beta_rms) * (grad**2)    # ค่าเฉลี่ยเคลื่อนที่ของกำลังสองเกรเดียนต์
    w_rmsprop = w_rmsprop - lr_rmsprop * grad / (np.sqrt(v_t_rms) + epsilon_rms)
    history_rmsprop.append(w_rmsprop)

# --- AdamW: ใช้ก้าวของ Adam บวกกับก้าวลดทอนค่าน้ำหนักที่แยกออกมาต่างหาก ---
w_adamw = w_init
m_t_w = 0.0
v_t_w = 0.0
beta1_w, beta2_w = 0.9, 0.999
epsilon_w = 1e-8
lambda_w = 0.01      # สัมประสิทธิ์การลดทอนค่าน้ำหนัก (ตัวอย่างค่านิยม)
lr_adamw = 0.5

history_adamw = [w_adamw]
for t in range(1, epochs + 1):
    grad = gradient(w_adamw)
    m_t_w = beta1_w * m_t_w + (1 - beta1_w) * grad
    v_t_w = beta2_w * v_t_w + (1 - beta2_w) * (grad**2)
    m_hat_w = m_t_w / (1 - beta1_w**t)     # แก้ความเอนเอียง
    v_hat_w = v_t_w / (1 - beta2_w**t)
    # ก้าวของ Adam บวกก้าวลดทอนค่าน้ำหนัก lambda*theta_old ที่ไม่ผ่านตัวหารจากโมเมนต์อันดับสอง
    w_adamw = w_adamw - lr_adamw * (m_hat_w / (np.sqrt(v_hat_w) + epsilon_w) + lambda_w * w_adamw)
    history_adamw.append(w_adamw)

plt.figure(figsize=(10, 5))
plt.plot(history_adagrad, label='AdaGrad', marker='^')
plt.plot(history_rmsprop, label='RMSprop', marker='v')
plt.plot(history_adamw, label='AdamW', marker='d')
plt.title("พฤติกรรมการลู่เข้าของ AdaGrad, RMSprop และ AdamW")
plt.xlabel("รอบการฝึก")
plt.ylabel("ค่าน้ำหนัก (เป้าหมาย = 0)")
plt.axhline(0, color='red', linestyle='--', label='จุดต่ำสุดสัมบูรณ์')
plt.legend()
plt.show()

### เปรียบเทียบเส้นทางของตัวปรับค่าทั้งหมด
รวมเส้นทางของ SGD, Momentum, Adam, AdaGrad, RMSprop และ AdamW จากจุดเริ่มต้นเดียวกันไว้ในกราฟเดียว (ตามหัวข้อ 4.4 การเปรียบเทียบตัวปรับค่า) เพื่อให้เห็นความแตกต่างของขนาดก้าวและพฤติกรรมระยะยาวของแต่ละวิธีโดยตรง

In [ ]:
# รวมเส้นทางของตัวปรับค่าทั้งหมดที่คำนวณไว้ข้างต้น (โจทย์ทดสอบเดียวกัน จุดเริ่มต้นเดียวกันทุกวิธี)
plt.figure(figsize=(11, 6))
plt.plot(history_sgd, label='SGD', marker='o', markersize=4)
plt.plot(history_momentum, label='Momentum', marker='x', markersize=4)
plt.plot(history_adam, label='Adam', marker='s', markersize=4)
plt.plot(history_adagrad, label='AdaGrad', marker='^', markersize=4)
plt.plot(history_rmsprop, label='RMSprop', marker='v', markersize=4)
plt.plot(history_adamw, label='AdamW', marker='d', markersize=4)
plt.title("เปรียบเทียบเส้นทางการหาค่าเหมาะที่สุดของตัวปรับค่าทั้งหมด")
plt.xlabel("รอบการฝึก")
plt.ylabel("ค่าน้ำหนัก (เป้าหมาย = 0)")
plt.axhline(0, color='red', linestyle='--', label='จุดต่ำสุดสัมบูรณ์')
plt.legend()
plt.show()

print(f"ค่าน้ำหนักสุดท้ายหลังฝึกครบ {epochs} รอบ:")
for name, hist in [('SGD', history_sgd), ('Momentum', history_momentum), ('Adam', history_adam),
                   ('AdaGrad', history_adagrad), ('RMSprop', history_rmsprop), ('AdamW', history_adamw)]:
    print(f"  {name}: {hist[-1]:.4f}")

## 7. ปัญหาเกี่ยวกับขนาดการไล่ระดับ (Vanishing / Exploding Gradient)
หัวข้อ 4.5: สาธิตสิ่งที่เกิดขึ้นเมื่อเกรเดียนต์ผ่านหลายชั้นด้วย Sigmoid เทียบกับ ReLU


In [ ]:
def relu_derivative(x):
    return np.where(x > 0, 1, 0)

# จำลองเกรเดียนต์เริ่มต้นที่ไหลมาจากเอาต์พุต (กำหนดค่าเริ่มต้นเป็น 1.0)
initial_grad = 1.0

# กำหนดค่าสุ่มตั้งต้นให้ผลลัพธ์ทำซ้ำได้เสมอทุกครั้งที่รัน
# (เลือกค่านี้เพราะช่วง [-3, 3] ครอบคลุมค่าลบ หากสุ่มได้ค่าลบใน inputs_relu
# อนุพันธ์ ReLU ของค่านั้นจะเป็น 0 พอดี ทำให้เกรเดียนต์สะสมกลายเป็น 0 ทันทีตั้งแต่ชั้นแรก ๆ
# จนกราฟไม่แสดงพฤติกรรมการรักษาสัญญาณของ ReLU ตามที่ตั้งใจสาธิต)
np.random.seed(2553)

# สุ่มข้อมูลอินพุตในช่วงต่างๆ
inputs_sigmoid = np.random.uniform(-3, 3, 10)
inputs_relu = np.random.uniform(-3, 3, 10)

grad_flow_sigmoid = [initial_grad]
grad_flow_relu = [initial_grad]

curr_grad_sig = initial_grad
curr_grad_relu = initial_grad

print("จำลองการไหลย้อนของเกรเดียนต์ผ่าน 10 ชั้น (ซิมิวเลเตอร์):")
for i in range(10):
    curr_grad_sig = curr_grad_sig * sigmoid_derivative(inputs_sigmoid[i])
    grad_flow_sigmoid.append(curr_grad_sig)

    curr_grad_relu = curr_grad_relu * relu_derivative(inputs_relu[i])
    grad_flow_relu.append(curr_grad_relu)

plt.figure(figsize=(10, 5))
plt.plot(grad_flow_sigmoid, label='การไหลของเกรเดียนต์ (Sigmoid)', marker='v')
plt.plot(grad_flow_relu, label='การไหลของเกรเดียนต์ (ReLU)', marker='^')
plt.yscale('log')
plt.title("ปัญหาเกรเดียนต์สูญหาย: Sigmoid เทียบกับ ReLU (แกน Y แบบ Log Scale)")
plt.xlabel("จำนวนชั้นที่ผ่าน (ย้อนกลับจากเอาต์พุตสู่อินพุต)")
plt.ylabel("ขนาดของเกรเดียนต์ (Log)")
plt.legend()
plt.show()

print("บทสรุป: จะสังเกตได้ว่าเกรเดียนต์ฝั่ง Sigmoid หดตัวหายไปเข้าใกล้ 0 อย่างรวดเร็วมาก (ลู่ลงด้านล่างของกราฟ Log) เพราะอนุพันธ์สูงสุดไม่เกิน 0.25 ในขณะที่ ReLU รักษาสัญญาณได้ดีกว่า")

## บทสรุป
Notebook นี้เป็นส่วนขยายของบทที่ 4 ผู้อ่านสามารถเปลี่ยนค่าตัวแปร เช่น `learning_rate`, `epochs` หรือโครงสร้าง `W1`, `W2` เพื่อสังเกตผลที่ตามมา และนำไปปรับใช้ในอัลกอริทึมจริงต่อไปได้
